# Генерация синтетических данных с использованием Faker

In [1]:
!pip install faker

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.8 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 2.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\maxpl\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

In [3]:
# Инициализация генератора (русская локаль для реалистичных ФИО, адресов и т.д.)
fake = Faker('ru_RU')
Faker.seed(42)      # фиксируем seed для воспроизводимости
random.seed(42)

## Генерация синтетических данных для варианта "Сотрудники компании"

In [4]:
# DataFrame с отделами
def generate_departments(n):
   
    departments = []
    for i in range(1, n + 1):
        departments.append({
            'department_id': i,
            'name': fake.unique.job() + ' отдел',  # уникальное название
            'location': fake.city()
        })
    return pd.DataFrame(departments)

# DataFrame с должностями и диапазоном зарплат
def generate_positions(n: int) -> pd.DataFrame:
    
    positions = []
    for i in range(1, n + 1):
        min_salary = random.randint(30000, 60000)
        max_salary = min_salary + random.randint(20000, 80000)
        positions.append({
            'position_id': i,
            'name': fake.unique.job(),
            'min_salary': min_salary,
            'max_salary': max_salary
        })
    return pd.DataFrame(positions)

# генерируем сотрудников со случайными атрибутами
def generate_employees(n: int, department_ids: list, position_ids: list) -> pd.DataFrame:
    
    employees = []
    for emp_id in range(1, n + 1):        
        first_name = fake.first_name()
        last_name = fake.last_name()
        patronymic = fake.middle_name()
        birth_date = fake.date_of_birth(minimum_age=18, maximum_age=70)
        phone = fake.phone_number()
        email = fake.email()
        address = fake.address().replace('\n', ', ')
        
        # дата найма: не раньше 18-летия и не позже сегодняшнего дня
        min_hire_date = birth_date + timedelta(days=18*365)
        hire_date = fake.date_between(start_date=min_hire_date, end_date='today')
        
        # статус -  active / terminated (10% уволены)
        status = random.choices(['active', 'terminated'], weights=[0.9, 0.1])[0]
        
        # выбор отдела и должности
        department_id = random.choice(department_ids)
        position_id = random.choice(position_ids)
        
        # зарплата     
        salary = random.randint(40000, 150000)
        
        employees.append({
            'employee_id': emp_id,
            'last_name': last_name,
            'first_name': first_name,
            'patronymic': patronymic,
            'birth_date': birth_date,
            'gender': random.choice(['М', 'Ж']),
            'address': address,
            'phone': phone,
            'email': email,
            'hire_date': hire_date,
            'status': status,
            'department_id': department_id,
            'position_id': position_id,
            'salary': salary
        })
    return pd.DataFrame(employees)

# создать историю зарплат на основе данных сотрудников
def generate_salary_history(employees_df: pd.DataFrame, avg_records: int = 3):    
    history = []
    record_id = 1
    
    for _, emp in employees_df.iterrows():
        emp_id = emp['employee_id']
        hire_date = emp['hire_date']
        current_salary = emp['salary']
        
        # количество изменений (0 – если сотрудник только нанят и ещё не было изменений)
        num_changes = random.choices([0, 1, 2, 3, 4], weights=[0.2, 0.3, 0.3, 0.1, 0.1])[0]
        
        # генерируем даты изменений (после hire_date и до сегодня)
        change_dates = sorted([fake.date_between(start_date=hire_date, end_date='today') 
                               for _ in range(num_changes)])
        
        # начальная зарплата при найме (можно сделать немного отличающейся от текущей)
        #  будем считать, что первая запись – это зарплата при найме,
        # а последующие – повышения.
        if num_changes == 0:
            # Если изменений не было, всё равно добавим одну запись (начальная)
            history.append({
                'history_id': record_id,
                'employee_id': emp_id,
                'change_date': hire_date,
                'new_salary': current_salary
            })
            record_id += 1
        else:
            # генерируем возрастающие зарплаты
            salary_values = sorted([random.randint(30000, current_salary) for _ in range(num_changes)])
            # добавляем текущую зарплату как последнюю
            salary_values.append(current_salary)
            # даты: hire_date и change_dates
            all_dates = [hire_date] + change_dates
            for i in range(len(all_dates)):
                history.append({
                    'history_id': record_id,
                    'employee_id': emp_id,
                    'change_date': all_dates[i],
                    'new_salary': salary_values[i]
                })
                record_id += 1
    return pd.DataFrame(history)

In [5]:
N_DEPARTMENTS = 10
N_POSITIONS = 20
N_EMPLOYEES = 500

In [9]:
departments_df = generate_departments(N_DEPARTMENTS)
positions_df = generate_positions(N_POSITIONS)
employees_df = generate_employees(
    N_EMPLOYEES,
    departments_df['department_id'].tolist(),
    positions_df['position_id'].tolist()
)

In [10]:
departments_df

,department_id,name,location
0,1,Радист отдел,п. Джейрах
1,2,Прозектор отдел,с. Беслан
2,3,Предприниматель отдел,ст. Североморск
3,4,Тракторист отдел,к. Зарайск
4,5,Изобретатр отдел,клх Нарьян-Мар
5,6,Психиатр отдел,д. Яшалта
6,7,Психоневропатолог отдел,к. Мокшан
7,8,Оператор коллцентра отдел,д. Губкин
8,9,Педиатр отдел,г. Ростов-на-Дону
9,10,Прокурор отдел,п. Азов (Рост.)


In [11]:
positions_df

,position_id,name,min_salary,max_salary
0,1,Блоггер,43513,72523
1,2,Борт-механик,57608,118941
2,3,Инженер-строитель,37583,62037
3,4,Переплётчик,53172,83069
4,5,Сварщик,30299,64633
5,6,Мастер маникюра,46571,96456
6,7,Математик,42229,66190
7,8,Воздухоплаватель,50245,112039
8,9,Медник,51848,112252
9,10,Заточник,59909,111576


In [12]:
employees_df

,employee_id,last_name,first_name,patronymic,birth_date,gender,address,phone,email,hire_date,status,department_id,position_id,salary
0,1,Исаков,Станимир,Евгеньевна,1998-05-18,М,"с. Кизел, пр. Гончарова, д. 7 к. 30, 825431",8 821 444 88 96,nlazareva@example.net,2025-04-23,active,3,4,52624
1,2,Калашникова,Ерофей,Эдуардовна,1967-01-24,Ж,"д. Костомукша, наб. Пожарского, д. 93 к. 6/1, ...",8 (123) 123-9204,semenovnarkis@example.net,1997-05-06,active,10,17,73084
2,3,Кондратьев,Милий,Фомич,1977-02-18,Ж,"клх Усть-Камчатск, бул. Кузнечный, д. 1/9, 717217",+75847262905,julijatarasova@example.net,2018-07-06,active,7,3,88896
3,4,Афанасьев,Ермолай,Феофанович,2004-02-08,М,"ст. Муром, ш. Фабричное, д. 4 стр. 33, 539835",+7 (992) 052-2299,ostapbobilev@example.org,2024-06-15,active,4,8,79318
4,5,Лукин,Леон,Фёдорович,1963-08-13,Ж,"с. Шелехов, пр. Центральный, д. 39 стр. 2/3, 5...",8 (773) 264-30-95,ippolit97@example.net,1989-04-04,active,1,3,93120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Белоусова,Панфил,Григорьевна,1958-12-25,Ж,"к. Петропавловск-Камчатский, наб. Школьная, д....",+7 523 029 99 66,efimlebedev@example.net,2019-09-13,terminated,6,12,64105
496,497,Кулаков,Ювеналий,Григорьевич,1961-08-28,Ж,"д. Гремячинск (Бурят.), ул. Ручейная, д. 954 с...",8 (500) 423-1820,hbelozerova@example.org,1993-07-25,active,7,12,87851
497,498,Юдина,Фрол,Андреевна,2006-03-27,М,"к. Гусь-Хрустальный, наб. Гражданская, д. 3 ст...",+7 (455) 381-51-82,galaktion2016@example.org,2025-06-24,active,3,19,57188
498,499,Мартынов,Андроник,Антипович,1990-03-26,Ж,"клх Мурманск, ул. Геологическая, д. 5 стр. 5/1...",+7 969 544 87 61,agafon_48@example.net,2020-01-06,active,5,8,112789


In [13]:
# Генерация истории зарплат
salary_history_df = generate_salary_history(employees_df, avg_records=3)
salary_history_df

,history_id,employee_id,change_date,new_salary
0,1,1,2025-04-23,30319
1,2,1,2025-12-18,52624
2,3,2,1997-05-06,73084
3,4,3,2018-07-06,88541
4,5,3,2018-11-02,88896
...,...,...,...,...
1295,1296,498,2026-04-07,57188
1296,1297,499,2020-01-06,85797
1297,1298,499,2020-03-27,111690
1298,1299,499,2025-03-03,112789


In [14]:
# Сохранение в CSV
departments_df.to_csv('departments.csv', index=False)
positions_df.to_csv('positions.csv', index=False)
employees_df.to_csv('employees.csv', index=False)
salary_history_df.to_csv('salary_history.csv', index=False)

# Задание на самостоятельную работу
Создать синтетический набор данных, сохранить его в csv файлы, построить на основе данных онтологию, вывести онтограф.




### Вариант 7. Транспортные средства и штрафы
Сущности:
- Владелец ТС: id, ФИО, водительское удостоверение (серия, номер), адрес, телефон.
- Автомобиль: id, госномер, марка, модель, год выпуска, VIN, цвет, id владельца.
- Штраф: id, id автомобиля, дата, статья нарушения, сумма, статус (оплачен/не оплачен).
- Страховой полис: id, id автомобиля, компания, дата начала, дата окончания, стоимость.

Объём: 1000 владельцев, 1200 автомобилей (некоторые владеют несколькими), 5000 штрафов, 1500 полисов.

Онтология:
- Классы: Владелец, Автомобиль, Штраф, Полис.
- Связи: владелец владеет автомобилями, автомобиль имеет штрафы и страховки.



In [20]:
print(dir(fake))

['__annotations__', '__class__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_factories', '_factory_map', '_locales', '_map_provider_method', '_optional_proxy', '_select_factory', '_select_factory_choice', '_select_factory_distribution', '_unique_proxy', '_weights', 'aba', 'add_provider', 'address', 'administrative_unit', 'am_pm', 'android_platform_token', 'ascii_company_email', 'ascii_email', 'ascii_free_email', 'ascii_safe_email', 'bank', 'bank_country', 'bban', 'bic', 'binary', 'boolean', 'bothify', 'bs', 'building_number', 'businesses_inn', 'businesses_ogrn', 'cache_pattern', 'catch_phrase', 'century', 'checking_account', 

In [23]:
print(fake.vin())

WC0JU7NN75FG62241


In [16]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

# Инициализация генератора с русской локалью для реалистичных ФИО, адресов и номеров машин
fake = Faker('ru_RU')

# Фиксируем seed для того, чтобы при каждом запуске данные генерировались одинаково
Faker.seed(42)      
random.seed(42)

In [28]:
#  Владельцы ТС
def generate_owners(n) -> pd.DataFrame:
    owners = []
    for i in range(1, n + 1):
        owners.append({
            'owner_id': i,
            'full_name': fake.name(), # Случайное ФИО
            # Генерируем серию (4 цифры) и номер (6 цифр) прав
            'driver_license': f"{random.randint(1000, 9999)} {random.randint(100000, 999999)}",
            # Адрес с удалением переносов строк для аккуратного вида
            'address': fake.address().replace('\n', ', '),
            'phone': fake.phone_number()
        })
    return pd.DataFrame(owners)

# Автомобили
def generate_cars(n, owner_ids) -> pd.DataFrame:
    cars = []
    # Небольшой словарь реалистичных марок и моделей
    brands_models = {
        'Lada': ['Vesta', 'Granta', 'Niva'],
        'Toyota': ['Camry', 'Corolla', 'RAV4'],
        'Kia': ['Rio', 'Sportage', 'Optima'],
        'Hyundai': ['Solaris', 'Creta', 'Tucson'],
        'Volkswagen': ['Polo', 'Tiguan', 'Jetta', 'Golf']
    }
    
    for i in range(1, n + 1):
        brand = random.choice(list(brands_models.keys()))
        model = random.choice(brands_models[brand])
        
        cars.append({
            'car_id': i,
            'license_plate': fake.license_plate(), # Случайный российский госномер
            'brand': brand,
            'model': model,
            'year': random.randint(2000, 2026),    # Год выпуска
            'vin': fake.vin(),                     # Уникальный VIN-код
            'color': fake.color_name(),
            'owner_id': random.choice(owner_ids) #Выбираем случайного водителя
            
        })
    return pd.DataFrame(cars)

#  Штрафы
def generate_fines(n, car_ids) -> pd.DataFrame:
    fines = []
    # Виды нарушений и базовые суммы штрафов
    violations = [
        ("Превышение скорости (12.9 ч.2 КоАП)", 500),
        ("Проезд на запрещающий сигнал (12.12 ч.1 КоАП)", 1000),
        ("Нарушение правил парковки (12.19 КоАП)", 1500),
        ("Выезд на встречную полосу (12.15 ч.4 КоАП)", 5000),
        ("Непристегнутый ремень (12.6 КоАП)", 1000)
    ]
    
    for i in range(1, n + 1):
        violation, amount = random.choice(violations)
        fines.append({
            'fine_id': i,
            'car_id': random.choice(car_ids), # Привязываем штраф к случайному автомобилю
            'date': fake.date_between(start_date='-3y', end_date='today'), # Штрафы за последние 3 года
            'violation': violation, 
            'amount': amount, 
            
            'status': random.choices(['оплачен', 'не оплачен'], weights=[0.7, 0.3])[0] # 70% оплачено, 30% не оплачено
        })
    return pd.DataFrame(fines)

#  Страховые полисы 
def generate_policies(n, car_ids) -> pd.DataFrame:
    policies = []
    companies = ['Росгосстрах', 'АльфаСтрахование', 'Ингосстрах', 'Т-Страхование']
    
    for i in range(1, n + 1):
        # Полис мог быть оформлен в любой день за последние 2 года
        start_date = fake.date_between(start_date='-2y', end_date='today')
        # Действует ровно 1 год
        end_date = start_date + timedelta(days=365) 
        
        policies.append({
            'policy_id': i,
            'car_id': random.choice(car_ids),
            'company': random.choice(companies),
            'start_date': start_date,
            'end_date': end_date,
            'cost': random.randint(5000, 25000) # Случайная стоимость от 5 до 25 тыс. руб.
        })
    return pd.DataFrame(policies)

In [29]:
# Указываем требуемые в задании объемы данных
N_OWNERS = 1000
N_CARS = 1200
N_FINES = 5000
N_POLICIES = 1500

#  Генерируем владельцев
owners_df = generate_owners(N_OWNERS)

# Генерируем авто, передавая список ID владельцев для связи
cars_df = generate_cars(N_CARS, owners_df['owner_id'].tolist())

# Генерируем штрафы, привязывая их к ID существующих автомобилей
fines_df = generate_fines(N_FINES, cars_df['car_id'].tolist())

# Генерируем страховые полисы для автомобилей
policies_df = generate_policies(N_POLICIES, cars_df['car_id'].tolist())



In [31]:

owners_df.to_csv('owners.csv', index=False)
cars_df.to_csv('cars.csv', index=False)
fines_df.to_csv('fines.csv', index=False)
policies_df.to_csv('policies.csv', index=False)


In [32]:
owners_df

,owner_id,full_name,driver_license,address,phone
0,1,Ефремова Алла Богдановна,5839 518900,"ст. Оленек, пр. 8 Марта, д. 7, 450384",+7 (579) 652-60-78
1,2,Данилова Глафира Тимофеевна,2284 432760,"клх Норильск, пр. Производственный, д. 746 к. ...",8 518 638 18 73
2,3,Ильина Екатерина Степановна,2136 927843,"п. Сузун, ш. Кузнецкое, д. 4 стр. 7/7, 698967",+7 (296) 504-88-58
3,4,Наум Теймуразович Куликов,3615 216771,"п. Благовещенск (Амур.), наб. Клубная, д. 6, 4...",+70836843722
4,5,Ильина Анжела Ниловна,7987 615835,"к. Арсеньев, наб. Родниковая, д. 8, 341908",+7 (682) 805-49-20
...,...,...,...,...,...
995,996,Попова Иванна Валериевна,7653 514308,"к. Благовещенск (Амур.), наб. Ленина, д. 44 ст...",+7 (309) 739-78-54
996,997,Исаков Ювеналий Аверьянович,5994 602165,"к. Шерегеш, ул. Крупской, д. 633, 478188",+7 (562) 540-25-48
997,998,Полякова Зоя Степановна,2076 813238,"п. Сухиничи, ул. Павлика Морозова, д. 6 стр. 2...",+7 (150) 995-2489
998,999,Доброслав Владиславович Власов,2531 636908,"клх Печенга, пр. Халтурина, д. 995 стр. 6, 268707",8 201 259 58 10


In [33]:
cars_df.head()

,car_id,license_plate,brand,model,year,vin,color,owner_id
0,1,ХB120 88,Toyota,RAV4,2000,PCPB2TKC0GMHR8856,Индиго,465
1,2,AB561 95,Toyota,Camry,2023,WZP0RUGD7ZUDU6911,Античный Белый,588
2,3,УУ629 196,Lada,Niva,2022,XA750ZLB6GZJR7747,Темный хаки,90
3,4,BУ350 21,Volkswagen,Jetta,2012,7WGWEEYF7JWMS9127,Светло-розовый,338
4,5,BK036 84,Kia,Rio,2024,L9CX0640263TM3972,Светло-серый,421


In [34]:
fines_df.head()

,fine_id,car_id,date,violation,amount,status
0,1,578,2025-04-01,Превышение скорости (12.9 ч.2 КоАП),500,не оплачен
1,2,235,2025-03-12,Непристегнутый ремень (12.6 КоАП),1000,оплачен
2,3,605,2023-12-12,Превышение скорости (12.9 ч.2 КоАП),500,оплачен
3,4,190,2024-03-02,Выезд на встречную полосу (12.15 ч.4 КоАП),5000,не оплачен
4,5,699,2024-09-05,Непристегнутый ремень (12.6 КоАП),1000,оплачен


In [35]:
policies_df.head()

,policy_id,car_id,company,start_date,end_date,cost
0,1,83,Росгосстрах,2025-11-15,2026-11-15,15167
1,2,344,Ингосстрах,2026-01-09,2027-01-09,15558
2,3,556,Т-Страхование,2024-05-09,2025-05-09,8780
3,4,339,Росгосстрах,2025-03-10,2026-03-10,6193
4,5,904,АльфаСтрахование,2024-06-26,2025-06-26,7889
